In [ ]:
!pip install pandas numpy scikit-learn torch matplotlib joblib

In [ ]:
import os
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

torch.set_num_threads(1)

print("Среда готова")


In [ ]:
# CNN-LSTM прогнозирует эксплуатационные и диагностические параметры
# следующего периода. Износ и остаточный ресурс рассчитываются отдельно
# по линейному правилу накопления повреждений Минера.

STATIC_COLUMNS = []

FORECAST_COLUMNS = [
    "cycles_total",
    "load_lt_40",
    "load_40_80",
    "load_80_100",
    "load_gt_100",
    "overload_cycles",
    "load_distribution_coef",
    "hoist_hours",
    "boom_angle_hours",
    "boom_length_hours",
    "rotation_hours",
    "broken_wires_count",
    "diameter_loss_percent",
    "corrosion_score",
    "abrasion_score",
    "deformation_score",
    "lubrication_score",
]

FEATURE_COLUMNS = FORECAST_COLUMNS.copy()
TARGET_COLUMNS = FORECAST_COLUMNS.copy()

WINDOW_SIZE = 5
MINER_N_REF = 8000
MINER_M = 3

INPUT_SIZE = len(FEATURE_COLUMNS)
OUTPUT_SIZE = len(TARGET_COLUMNS)

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Настройки готовы")
print("Количество входных признаков:", INPUT_SIZE)
print("Количество прогнозируемых параметров:", OUTPUT_SIZE)


In [ ]:
def calculate_load_distribution_coef(load_lt_40, load_40_80, load_80_100, load_gt_100):
    """
    Коэффициент распределения нагрузок.
    Чем больше доля тяжёлых и перегрузочных режимов, тем выше коэффициент.
    """
    return (
        0.25 * load_lt_40 +
        0.60 * load_40_80 +
        0.90 * load_80_100 +
        1.10 * load_gt_100
    )


def calculate_visual_condition_index(features):
    """
    Индекс визуального состояния троса.
    Чем больше значение, тем хуже визуальное состояние.
    """
    broken_wires_norm = min(features["broken_wires_count"] / 20, 1)
    diameter_loss_norm = min(features["diameter_loss_percent"] / 10, 1)

    corrosion_norm = min(features["corrosion_score"] / 5, 1)
    abrasion_norm = min(features["abrasion_score"] / 5, 1)
    deformation_norm = min(features["deformation_score"] / 5, 1)
    lubrication_norm = min(features["lubrication_score"] / 5, 1)

    visual_index = (
        0.25 * broken_wires_norm +
        0.25 * diameter_loss_norm +
        0.15 * corrosion_norm +
        0.15 * abrasion_norm +
        0.15 * deformation_norm +
        0.05 * lubrication_norm
    )

    return visual_index


def calculate_damage_by_miner(features, n_ref=MINER_N_REF, m=MINER_M):
    """
    Расчёт прироста накопленной повреждённости по линейному правилу Минера.

    D = Σ(n_i / N_i)

    где:
    n_i — число циклов в i-м диапазоне нагрузки;
    N_i — допустимое число циклов до предельного состояния при этой нагрузке;
    D — накопленная повреждённость.

    Нейросеть прогнозирует эксплуатационные и диагностические параметры,
    а степень износа и остаточный ресурс рассчитываются на основе D.
    """
    cycles_total = max(0, features["cycles_total"])

    # Условные коэффициенты нагрузки для укрупнённых диапазонов.
    # Они соответствуют средней относительной нагрузке в диапазоне.
    load_ranges = [
        ("load_lt_40", 0.30),     # до 40 %
        ("load_40_80", 0.60),     # 40–80 %
        ("load_80_100", 0.90),    # 80–100 %
        ("load_gt_100", 1.10),    # более 100 %
    ]

    delta_damage = 0.0

    for column, load_coef in load_ranges:
        load_share = max(0, min(1, features[column]))
        cycles_in_range = cycles_total * load_share

        # Чем выше нагрузка, тем меньше допустимое число циклов.
        allowable_cycles = n_ref / (load_coef ** m)

        if allowable_cycles > 0:
            delta_damage += cycles_in_range / allowable_cycles

    # Перегрузочные циклы уже учтены в диапазоне load_gt_100,
    # поэтому повторно не добавляются в сумму Минера.

    # Визуальный осмотр используется как поправочный коэффициент.
    # Чем хуже визуальное состояние, тем быстрее накапливается повреждённость.
    visual_index = calculate_visual_condition_index(features)
    visual_factor = 1 + 0.35 * visual_index

    delta_damage = delta_damage * visual_factor

    return max(0, delta_damage)


def calculate_wear_by_miner(previous_damage, features, n_ref=MINER_N_REF, m=MINER_M):
    """
    Накопительный расчёт износа и остаточного ресурса по правилу Минера.

    D_new = D_prev + ΔD
    W = 100 · min(D_new, 1)
    R = 100 · (1 − min(D_new, 1))

    где:
    D_new — накопленная повреждённость;
    W — степень износа, %;
    R — остаточный ресурс, %.
    """
    delta_damage = calculate_damage_by_miner(
        features=features,
        n_ref=n_ref,
        m=m
    )

    new_damage = max(0, previous_damage + delta_damage)

    wear_percent = 100 * min(new_damage, 1)
    remaining_resource = 100 * (1 - min(new_damage, 1))

    return new_damage, wear_percent, remaining_resource


def get_engineering_status(remaining_resource):
    if remaining_resource > 60:
        return "Ресурс достаточный"
    elif remaining_resource > 40:
        return "Требуется плановый контроль"
    elif remaining_resource > 20:
        return "Требуется усиленный контроль"
    else:
        return "Приближение к замене"


def postprocess_predicted_features(values):
    """
    Приводит выход нейросети к физически допустимым значениям.
    """
    features = {column: float(value) for column, value in zip(TARGET_COLUMNS, values)}

    # Доли нагрузки ограничиваем и нормируем
    load_columns = ["load_lt_40", "load_40_80", "load_80_100", "load_gt_100"]

    for column in load_columns:
        features[column] = max(0, min(1, features[column]))

    load_sum = sum(features[column] for column in load_columns)

    if load_sum > 0:
        for column in load_columns:
            features[column] = features[column] / load_sum
    else:
        features["load_lt_40"] = 1.0
        features["load_40_80"] = 0.0
        features["load_80_100"] = 0.0
        features["load_gt_100"] = 0.0

    # Неотрицательные величины
    non_negative_columns = [
        "cycles_total",
        "overload_cycles",
        "hoist_hours",
        "boom_angle_hours",
        "boom_length_hours",
        "rotation_hours",
        "broken_wires_count",
        "diameter_loss_percent"
    ]

    for column in non_negative_columns:
        features[column] = max(0, features[column])

    # Балльные показатели визуального осмотра
    score_columns = [
        "corrosion_score",
        "abrasion_score",
        "deformation_score",
        "lubrication_score"
    ]

    for column in score_columns:
        features[column] = max(0, min(5, features[column]))

    # Коэффициент нагрузки пересчитываем из нормированных долей
    features["load_distribution_coef"] = calculate_load_distribution_coef(
        features["load_lt_40"],
        features["load_40_80"],
        features["load_80_100"],
        features["load_gt_100"]
    )

    return features


print("Расчётные функции готовы")


In [ ]:
# Загружаем подготовленный синтетический датасет.
# Файл должен находиться рядом с ноутбуком. В среде ChatGPT он также
# может находиться в каталоге /mnt/data.

candidate_paths = [
    Path("crane_training_dataset_1080.csv"),
    Path("/mnt/data/crane_training_dataset_1080.csv"),
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Файл crane_training_dataset_1080.csv не найден. "
        "Поместите его рядом с ноутбуком."
    )

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df["period_start"] = pd.to_datetime(df["period_start"], errors="raise")
df = df.sort_values(["crane_id", "period_start"]).reset_index(drop=True)

print("Датасет загружен:", DATA_PATH)
print("Размер таблицы:", df.shape)
print("Количество кранов:", df["crane_id"].nunique())
print(df["dataset_split"].value_counts())

df.head()


In [ ]:
df.info()

In [ ]:
df.head(10)

In [ ]:
# Визуальная проверка динамики износа по правилу Минера и остаточного ресурса

df["period_start"] = pd.to_datetime(df["period_start"])

crane_1 = df[df["crane_id"] == 1].sort_values("period_start")

plt.figure(figsize=(11, 5))

plt.plot(
    crane_1["period_start"],
    crane_1["wear_percent"],
    marker="o",
    linewidth=2,
    label="Степень износа по Минеру, %"
)

plt.plot(
    crane_1["period_start"],
    crane_1["remaining_resource"],
    marker="o",
    linewidth=2,
    label="Остаточный ресурс по Минеру, %"
)

plt.xlabel("Период эксплуатации")
plt.ylabel("Значение, %")
plt.title("Динамика износа и остаточного ресурса троса по правилу Минера")
plt.legend()
plt.grid(True)

plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("models/wear_resource_dynamics.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Проверяем структуру датасета и обучаем нормализаторы только на train-части.

required_columns = list(dict.fromkeys(
    FEATURE_COLUMNS
    + TARGET_COLUMNS
    + [
        "crane_id",
        "period_start",
        "dataset_split",
        "damage_index",
        "wear_percent",
        "remaining_resource",
    ]
))

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"В таблице отсутствуют столбцы: {missing_columns}")

df[FEATURE_COLUMNS] = (
    df[FEATURE_COLUMNS]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
)

train_rows = df[df["dataset_split"] == "train"]

if train_rows.empty:
    raise ValueError("В датасете отсутствуют строки dataset_split='train'")

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_x.fit(train_rows[FEATURE_COLUMNS])
scaler_y.fit(train_rows[TARGET_COLUMNS])

joblib.dump(scaler_x, "models/scaler_x.pkl")
joblib.dump(scaler_y, "models/scaler_y.pkl")

print("Нормализаторы обучены только на train-части")


In [ ]:
def create_sequences_for_split(df, split_name, scaler_x, scaler_y, window_size):
    X, y = [], []
    meta_rows = []

    split_df = df[df["dataset_split"] == split_name].copy()

    for crane_id, crane_df in split_df.groupby("crane_id"):
        crane_df = crane_df.sort_values("period_start").reset_index(drop=True)

        if len(crane_df) <= window_size:
            continue

        x_crane = scaler_x.transform(crane_df[FEATURE_COLUMNS])
        y_crane = scaler_y.transform(crane_df[TARGET_COLUMNS])

        for i in range(len(crane_df) - window_size):
            target_row = crane_df.iloc[i + window_size]
            previous_row = crane_df.iloc[i + window_size - 1]

            X.append(x_crane[i:i + window_size])
            y.append(y_crane[i + window_size])

            meta_rows.append({
                "crane_id": int(crane_id),
                "target_period_start": target_row["period_start"],
                "previous_damage": float(previous_row["damage_index"]),
                "fact_damage": float(target_row["damage_index"]),
                "fact_wear": float(target_row["wear_percent"]),
                "fact_resource": float(target_row["remaining_resource"]),
            })

    if not X:
        raise ValueError(
            f"Для выборки {split_name} не сформировано ни одного окна"
        )

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        pd.DataFrame(meta_rows),
    )


In [ ]:
# Формируем временные окна отдельно для train, validation и test.
# Данные одного крана не пересекаются между выборками.

x_train, y_train, meta_train = create_sequences_for_split(
    df, "train", scaler_x, scaler_y, WINDOW_SIZE
)
x_val, y_val, meta_val = create_sequences_for_split(
    df, "validation", scaler_x, scaler_y, WINDOW_SIZE
)
x_test, y_test, meta_test = create_sequences_for_split(
    df, "test", scaler_x, scaler_y, WINDOW_SIZE
)

print("Train:", x_train.shape, y_train.shape)
print("Validation:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)


In [ ]:
class CraneDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x_data = torch.tensor(x_data, dtype=torch.float32)
        self.y_data = torch.tensor(y_data, dtype=torch.float32)

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]


train_dataset = CraneDataset(x_train, y_train)
val_dataset = CraneDataset(x_val, y_val)
test_dataset = CraneDataset(x_test, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Dataset и DataLoader созданы")

batch_x, batch_y = next(iter(train_loader))

print("batch_x:", batch_x.shape)
print("batch_y:", batch_y.shape)

In [ ]:
class CNNLSTMModel(nn.Module):
    def __init__(self, input_size, output_size, cnn_channels=32, lstm_hidden=64):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(
                in_channels=input_size,
                out_channels=cnn_channels,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.BatchNorm1d(cnn_channels)
        )

        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        # x: [batch, window_size, features]

        # Для CNN нужно: [batch, features, window_size]
        x = x.permute(0, 2, 1)

        x = self.cnn(x)

        # Для LSTM нужно: [batch, window_size, channels]
        x = x.permute(0, 2, 1)

        lstm_out, _ = self.lstm(x)

        # Берём последнее состояние временной последовательности
        last_state = lstm_out[:, -1, :]

        output = self.fc(last_state)

        return output


model = CNNLSTMModel(
    input_size=len(FEATURE_COLUMNS),
    output_size=len(TARGET_COLUMNS)
)

print(model)

test_output = model(batch_x)

print("test_output:", test_output.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Используемое устройство:", device)

model = model.to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float("inf")
epochs_without_improvement = 0
EARLY_STOPPING_PATIENCE = 20

train_losses = []
val_losses = []

EPOCHS = 150

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= max(len(train_loader), 1)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item()

    val_loss /= max(len(val_loader), 1)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "models/cnn_lstm_model.pt")
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={val_loss:.6f}"
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            "Ранняя остановка: validation loss не улучшалась "
            f"{EARLY_STOPPING_PATIENCE} эпох"
        )
        break

print("Обучение завершено")
print("Лучшая ошибка на validation:", best_val_loss)


In [ ]:
model.load_state_dict(torch.load("models/cnn_lstm_model.pt", map_location=device))
model.eval()

print("Лучшая модель загружена")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label="Ошибка на обучении")
plt.plot(val_losses, label="Ошибка на проверке")

plt.xlabel("Эпоха")
plt.ylabel("MSE")
plt.title("Динамика ошибки обучения CNN-LSTM")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("models/training_loss_cnn_lstm.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
model = model.to(device)

model.load_state_dict(
    torch.load("models/cnn_lstm_model.pt", map_location=device)
)

model.eval()

predictions_list = []
targets_list = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)

        predictions_batch = model(batch_x).cpu().numpy()

        predictions_list.append(predictions_batch)
        targets_list.append(batch_y.numpy())

predictions_scaled = np.vstack(predictions_list)
targets_scaled = np.vstack(targets_list)

# Возвращаем прогнозируемые параметры из нормализованного масштаба
predictions_raw = scaler_y.inverse_transform(predictions_scaled)
targets_raw = scaler_y.inverse_transform(targets_scaled)

# Приводим прогноз к физически допустимым значениям
predicted_features_list = [
    postprocess_predicted_features(row)
    for row in predictions_raw
]

predictions_features_df = pd.DataFrame(predicted_features_list)
targets_features_df = pd.DataFrame(targets_raw, columns=TARGET_COLUMNS)

print("Проверка выполнена")
print("Размер прогноза параметров:", predictions_features_df.shape)
print("Размер фактических параметров:", targets_features_df.shape)

predictions_features_df.head()

In [ ]:
# Метрики качества для прогнозируемых эксплуатационных и диагностических параметров

metrics_rows = []

for column in TARGET_COLUMNS:
    mae = mean_absolute_error(targets_features_df[column], predictions_features_df[column])
    rmse = np.sqrt(mean_squared_error(targets_features_df[column], predictions_features_df[column]))

    metrics_rows.append({
        "parameter": column,
        "MAE": mae,
        "RMSE": rmse
    })

feature_metrics_df = pd.DataFrame(metrics_rows)

feature_metrics_df.to_csv("results/forecast_feature_metrics.csv", index=False)

print("Среднее MAE по прогнозируемым параметрам:", round(feature_metrics_df["MAE"].mean(), 4))
print("Среднее RMSE по прогнозируемым параметрам:", round(feature_metrics_df["RMSE"].mean(), 4))

feature_metrics_df

In [ ]:
# После прогноза параметров рассчитываем повреждённость, износ и остаточный ресурс по правилу Минера

calculated_rows = []

for i in range(len(predictions_features_df)):
    previous_damage = float(meta_test.loc[i, "previous_damage"])

    predicted_features = predictions_features_df.iloc[i].to_dict()
    target_features = targets_features_df.iloc[i].to_dict()

    predicted_damage, predicted_wear, predicted_resource = calculate_wear_by_miner(
        previous_damage=previous_damage,
        features=predicted_features
    )

    # Фактические расчётные значения из датасета.
    # В демонстрационном наборе они тоже получены по правилу Минера,
    # но на основе фактических, а не прогнозных параметров периода.
    fact_damage = float(meta_test.loc[i, "fact_damage"])
    fact_wear = float(meta_test.loc[i, "fact_wear"])
    fact_resource = float(meta_test.loc[i, "fact_resource"])

    calculated_rows.append({
        "crane_id": meta_test.loc[i, "crane_id"],
        "target_period_start": meta_test.loc[i, "target_period_start"],

        "previous_damage": previous_damage,
        "fact_damage": fact_damage,
        "predicted_damage": predicted_damage,
        "damage_error": abs(fact_damage - predicted_damage),

        "fact_wear": fact_wear,
        "predicted_wear": predicted_wear,
        "wear_error": abs(fact_wear - predicted_wear),

        "fact_resource": fact_resource,
        "predicted_resource": predicted_resource,
        "resource_error": abs(fact_resource - predicted_resource),

        "fact_cycles_total": target_features["cycles_total"],
        "predicted_cycles_total": predicted_features["cycles_total"],

        "fact_load_gt_100": target_features["load_gt_100"],
        "predicted_load_gt_100": predicted_features["load_gt_100"],

        "fact_broken_wires_count": target_features["broken_wires_count"],
        "predicted_broken_wires_count": predicted_features["broken_wires_count"],

        "fact_diameter_loss_percent": target_features["diameter_loss_percent"],
        "predicted_diameter_loss_percent": predicted_features["diameter_loss_percent"]
    })

result_df = pd.DataFrame(calculated_rows)
result_df = result_df.round(3)

result_df.to_csv("results/test_predictions.csv", index=False)

mae_wear = mean_absolute_error(result_df["fact_wear"], result_df["predicted_wear"])
rmse_wear = np.sqrt(mean_squared_error(result_df["fact_wear"], result_df["predicted_wear"]))

mae_resource = mean_absolute_error(result_df["fact_resource"], result_df["predicted_resource"])
rmse_resource = np.sqrt(mean_squared_error(result_df["fact_resource"], result_df["predicted_resource"]))

print("Износ троса по правилу Минера:")
print("MAE:", round(mae_wear, 3), "%")
print("RMSE:", round(rmse_wear, 3), "%")

print()

print("Остаточный ресурс по правилу Минера:")
print("MAE:", round(mae_resource, 3), "%")
print("RMSE:", round(rmse_resource, 3), "%")

result_df.head(10)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    result_df["fact_wear"].values,
    marker="o",
    linewidth=2,
    label="Фактический износ по Минеру"
)

plt.plot(
    result_df["predicted_wear"].values,
    marker="o",
    linewidth=2,
    label="Износ по прогнозным параметрам"
)

plt.xlabel("Номер тестового примера")
plt.ylabel("Износ троса, %")
plt.title("Сравнение износа по Минеру для фактических и прогнозных параметров")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("models/wear_prediction_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    result_df["fact_resource"].values,
    marker="o",
    linewidth=2,
    label="Фактический остаточный ресурс по Минеру"
)

plt.plot(
    result_df["predicted_resource"].values,
    marker="o",
    linewidth=2,
    label="Остаточный ресурс по прогнозным параметрам"
)

plt.xlabel("Номер тестового примера")
plt.ylabel("Остаточный ресурс троса, %")
plt.title("Сравнение остаточного ресурса по Минеру для фактических и прогнозных параметров")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig("models/resource_prediction_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
result_df["fact_status"] = result_df["fact_resource"].apply(get_engineering_status)
result_df["predicted_status"] = result_df["predicted_resource"].apply(get_engineering_status)

result_df.head(10)

In [ ]:
def predict_for_crane(crane_id):
    # Получаем историю выбранного крана
    crane_history = df[df["crane_id"] == crane_id].sort_values("period_start")

    if crane_history.empty:
        raise ValueError(f"Кран с id={crane_id} не найден")

    if len(crane_history) < WINDOW_SIZE:
        raise ValueError(
            f"Недостаточно данных для прогноза. "
            f"Нужно минимум {WINDOW_SIZE} периодов, найдено {len(crane_history)}"
        )

    # Берём последние периоды эксплуатации
    last_periods = crane_history.tail(WINDOW_SIZE)

    # Формируем входное окно
    x = last_periods[FEATURE_COLUMNS]
    x_scaled = scaler_x.transform(x)

    # Размерность: [1, window_size, features]
    x_tensor = torch.tensor(
        np.expand_dims(x_scaled, axis=0),
        dtype=torch.float32
    ).to(device)

    model.eval()

    with torch.no_grad():
        prediction_scaled = model(x_tensor).cpu().numpy()

    # Возвращаем прогноз параметров из нормализованного масштаба
    prediction_raw = scaler_y.inverse_transform(prediction_scaled)[0]

    predicted_features = postprocess_predicted_features(prediction_raw)

    # Предыдущая накопленная повреждённость по Минеру
    previous_damage = float(crane_history["damage_index"].iloc[-1])

    predicted_damage, predicted_wear, predicted_resource = calculate_wear_by_miner(
        previous_damage=previous_damage,
        features=predicted_features
    )

    status = get_engineering_status(predicted_resource)

    rounded_features = {
        key: round(value, 3)
        for key, value in predicted_features.items()
    }

    return {
        "crane_id": int(crane_id),
        "last_period": str(last_periods["period_start"].iloc[-1].date()),

        "predicted_features": rounded_features,

        "previous_damage_index": round(previous_damage, 4),
        "predicted_damage_index": round(predicted_damage, 4),
        "predicted_wear_percent": round(predicted_wear, 2),
        "predicted_remaining_resource": round(predicted_resource, 2),
        "status": status
    }


In [ ]:
prediction_result = predict_for_crane(1)

prediction_result

In [ ]:
print("Результат нейросетевого прогнозирования")
print("--------------------------------------")
print(f"Кран: {prediction_result['crane_id']}")
print(f"Последний известный период: {prediction_result['last_period']}")
print(f"Предыдущая накопленная повреждённость: {prediction_result['previous_damage_index']}")
print(f"Прогнозируемая накопленная повреждённость: {prediction_result['predicted_damage_index']}")
print(f"Прогнозируемая степень износа троса: {prediction_result['predicted_wear_percent']} %")
print(f"Прогнозируемый остаточный ресурс: {prediction_result['predicted_remaining_resource']} %")
print(f"Инженерный статус: {prediction_result['status']}")

print()
print("Часть прогнозируемых параметров следующего периода:")
for key in [
    "cycles_total",
    "load_gt_100",
    "overload_cycles",
    "broken_wires_count",
    "diameter_loss_percent"
]:
    print(f"{key}: {prediction_result['predicted_features'][key]}")


In [ ]:
all_predictions = []

for crane_id in df["crane_id"].unique():
    prediction = predict_for_crane(crane_id)
    all_predictions.append(prediction)

predictions_df = pd.DataFrame([
    {
        "crane_id": item["crane_id"],
        "last_period": item["last_period"],
        "previous_damage_index": item["previous_damage_index"],
        "predicted_damage_index": item["predicted_damage_index"],
        "predicted_wear_percent": item["predicted_wear_percent"],
        "predicted_remaining_resource": item["predicted_remaining_resource"],
        "status": item["status"],
        **{
            f"predicted_{key}": value
            for key, value in item["predicted_features"].items()
        }
    }
    for item in all_predictions
])

predictions_df


In [ ]:
predictions_df.to_csv(
    "results/crane_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Прогнозы сохранены в results/crane_predictions.csv")


In [ ]:
model_config = {
    "task_description": (
        "CNN-LSTM прогнозирует эксплуатационные и диагностические параметры "
        "следующего периода. Степень износа и остаточный ресурс рассчитываются "
        "после прогноза по линейному правилу накопления повреждений Минера."
    ),
    "window_size": WINDOW_SIZE,
    "static_columns": STATIC_COLUMNS,
    "forecast_columns": FORECAST_COLUMNS,
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "miner_n_ref": MINER_N_REF,
    "miner_m": MINER_M,
    "model_file": "cnn_lstm_model.pt",
    "scaler_x": "scaler_x.pkl",
    "scaler_y": "scaler_y.pkl"
}

with open("models/model_config.json", "w", encoding="utf-8") as file:
    json.dump(model_config, file, ensure_ascii=False, indent=4)

print("Конфигурация модели сохранена")
